# 04. 생활입지 피처 엔지니어링

서울 아파트 거래 데이터에 생활입지 피처를 추가한다.

최종 생성 대상:
- 최근접 응급의료기관 거리
- 최근접 대규모점포 거리
- 주요 일자리 Top 10까지의 최근접 거리

> 이 노트북은 **거리 기반 생활입지 피처 생성**에 집중한다.  
> 직장인구의 거래별 행정동 매칭·보정·밀도 피처는 최종 목적과 중복되어 제외한다.

## 0. 공통 설정

라이브러리, 데이터 경로, API 키와 공간거리 계산 함수를 준비한다.

In [1]:
# ============================================================
# 라이브러리 및 기본 설정
# ============================================================

import os
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from pyproj import Transformer
from sklearn.neighbors import BallTree

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

load_dotenv()


True

In [2]:
# ============================================================
# 경로 및 API 설정
# ============================================================

PROJECT_DIR = Path.cwd().parents[2]

DATA_ROOT = PROJECT_DIR / "data"
DATA_DIR = DATA_ROOT / "v2_5year_seoul"

PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_ROOT / "external"

DF_FE_BASE_PATH = PROCESSED_DIR / "df_fe_base.parquet"
LIVING_FEATURE_PATH = PROCESSED_DIR / "df_fe_living.parquet"

MART_RAW_PATH = EXTERNAL_DIR / "large_scale_store.csv"
WORKPLACE_PATH = EXTERNAL_DIR / "workplace_population.csv"
ADMIN_AREA_PATH = EXTERNAL_DIR / "서울시 상권분석서비스(영역-행정동).csv"

EMERGENCY_API_KEY = os.environ.get("EMERGENCY_API_KEY", "")
KAKAO_REST_API_KEY = os.environ.get("KAKAO_REST_API_KEY", "")

missing_keys = [
    key_name
    for key_name, key_value in {
        "EMERGENCY_API_KEY": EMERGENCY_API_KEY,
        "KAKAO_REST_API_KEY": KAKAO_REST_API_KEY,
    }.items()
    if not key_value
]

if missing_keys:
    raise ValueError(
        f".env 파일에 API 키가 없습니다: {', '.join(missing_keys)}"
    )

HOSPITAL_API_URL = (
    "https://apis.data.go.kr/"
    "B552657/ErmctInfoInqireService/"
    "getEgytListInfoInqire"
)

print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("API 키 로드 완료")


PROJECT_DIR: C:\Users\UK\Apartment price
API 키 로드 완료


In [3]:
# ============================================================
# 공통 함수
# ============================================================

EARTH_RADIUS_M = 6_371_000


def get_valid_seoul_coord_mask(df, lat_col="위도", lon_col="경도"):
    return (
        df[lat_col].between(37.4, 37.8)
        & df[lon_col].between(126.7, 127.3)
    )


def get_nearest_facility(source_lat, source_lon, target_lat, target_lon):
    source_coords = np.radians(
        np.column_stack([source_lat, source_lon])
    )
    target_coords = np.radians(
        np.column_stack([target_lat, target_lon])
    )

    tree = BallTree(target_coords, metric="haversine")
    distances, indices = tree.query(source_coords, k=1)

    return indices[:, 0], distances[:, 0] * EARTH_RADIUS_M


def clean_address(address):
    if pd.isna(address):
        return None

    address = str(address).strip()
    address = re.sub(r"\([^)]*\)", "", address)
    address = re.sub(r"\s+", " ", address)
    address = address.split(",")[0].strip()

    return address or None


def geocode_address(address, kakao_api_key):
    if not address:
        return None, None

    response = requests.get(
        "https://dapi.kakao.com/v2/local/search/address.json",
        headers={"Authorization": f"KakaoAK {kakao_api_key}"},
        params={"query": address},
        timeout=10,
    )
    response.raise_for_status()

    documents = response.json().get("documents", [])
    if not documents:
        return None, None

    result = documents[0]
    return float(result["y"]), float(result["x"])


def geocode_keyword(keyword, kakao_api_key):
    response = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {kakao_api_key}"},
        params={"query": keyword},
        timeout=10,
    )
    response.raise_for_status()

    documents = response.json().get("documents", [])
    if not documents:
        return None, None

    seoul_documents = [
        doc
        for doc in documents
        if (
            str(doc.get("road_address_name", "")).startswith("서울")
            or str(doc.get("address_name", "")).startswith("서울")
        )
    ]

    result = seoul_documents[0] if seoul_documents else documents[0]
    return float(result["y"]), float(result["x"])


In [4]:
# ============================================================
# 기본 피처 데이터 불러오기
# ============================================================

df_living = pd.read_parquet(DF_FE_BASE_PATH).copy()

apt_coord_mask = (
    df_living["위도"].notna()
    & df_living["경도"].notna()
)

print("거래 데이터:", df_living.shape)
print("아파트 좌표 결측:", (~apt_coord_mask).sum())
print(
    "transaction_id 중복:",
    df_living["transaction_id"].duplicated().sum(),
)


거래 데이터: (234302, 69)
아파트 좌표 결측: 0
transaction_id 중복: 0


# 1. 응급의료기관

서울 응급의료기관을 수집·정제한 뒤 각 아파트에서 가장 가까운 응급의료기관까지의 거리를 계산한다.

In [5]:
# ============================================================
# 서울 응급의료기관 수집
# ============================================================

params = {
    "serviceKey": EMERGENCY_API_KEY,
    "Q0": "서울특별시",
    "pageNo": 1,
    "numOfRows": 1000,
}

response = requests.get(
    HOSPITAL_API_URL,
    params=params,
    timeout=30,
)
response.raise_for_status()

root = ET.fromstring(response.content)

hospital_records = [
    {
        "병원ID": item.findtext("hpid"),
        "병원명": item.findtext("dutyName"),
        "응급의료기관코드": item.findtext("dutyEmcls"),
        "응급의료기관분류": item.findtext("dutyEmclsName"),
        "주소": item.findtext("dutyAddr"),
        "위도": item.findtext("wgs84Lat"),
        "경도": item.findtext("wgs84Lon"),
    }
    for item in root.findall(".//item")
]

hospital_df = pd.DataFrame(hospital_records)

print("API 기관 수:", int(root.findtext(".//totalCount", default="0")))
print("수집 행 수:", len(hospital_df))


API 기관 수: 74
수집 행 수: 74


In [6]:
# ============================================================
# 응급의료기관 정제
# ============================================================

hospital_clean = hospital_df.copy()

hospital_clean["위도"] = pd.to_numeric(
    hospital_clean["위도"],
    errors="coerce",
)
hospital_clean["경도"] = pd.to_numeric(
    hospital_clean["경도"],
    errors="coerce",
)

hospital_clean = (
    hospital_clean
    .drop_duplicates(subset="병원ID", keep="last")
    .reset_index(drop=True)
)

hospital_valid_coord = get_valid_seoul_coord_mask(hospital_clean)

hospital_valid = (
    hospital_clean.loc[hospital_valid_coord]
    .reset_index(drop=True)
)

print("정제 후 병원 수:", len(hospital_clean))
print("정상 서울 좌표:", len(hospital_valid))
print("비정상·결측 좌표:", (~hospital_valid_coord).sum())

print("\n응급의료기관 분류:")
print(
    hospital_valid["응급의료기관분류"]
    .value_counts(dropna=False)
)


정제 후 병원 수: 74
정상 서울 좌표: 74
비정상·결측 좌표: 0

응급의료기관 분류:
응급의료기관분류
지역응급의료센터     24
응급실운영신고기관    23
지역응급의료기관     20
권역응급의료센터      7
Name: count, dtype: int64


In [7]:
# ============================================================
# 최근접 응급의료기관 피처 생성
# ============================================================

nearest_idx, nearest_distance = get_nearest_facility(
    df_living.loc[apt_coord_mask, "위도"].to_numpy(),
    df_living.loc[apt_coord_mask, "경도"].to_numpy(),
    hospital_valid["위도"].to_numpy(),
    hospital_valid["경도"].to_numpy(),
)

nearest_hospitals = (
    hospital_valid.iloc[nearest_idx]
    .reset_index(drop=True)
)

df_living.loc[
    apt_coord_mask,
    "최근접응급의료기관",
] = nearest_hospitals["병원명"].to_numpy()

df_living.loc[
    apt_coord_mask,
    "최근접응급의료기관분류",
] = nearest_hospitals["응급의료기관분류"].to_numpy()

df_living.loc[
    apt_coord_mask,
    "응급의료기관거리_m",
] = nearest_distance

df_living["응급의료기관거리_km"] = (
    df_living["응급의료기관거리_m"] / 1000
)


In [8]:
# ============================================================
# 응급의료기관 피처 검증
# ============================================================

hospital_feature_cols = [
    "최근접응급의료기관",
    "최근접응급의료기관분류",
    "응급의료기관거리_m",
    "응급의료기관거리_km",
]

print("결측치:")
print(df_living[hospital_feature_cols].isna().sum())

print("\n거리 통계(km):")
print(df_living["응급의료기관거리_km"].describe())


결측치:
최근접응급의료기관      0
최근접응급의료기관분류    0
응급의료기관거리_m     0
응급의료기관거리_km    0
dtype: int64

거리 통계(km):
count    234302.000000
mean          1.239971
std           0.688347
min           0.029376
25%           0.732115
50%           1.164783
75%           1.624423
max           4.626382
Name: 응급의료기관거리_km, dtype: float64


# 2. 대규모점포

현재 정상 영업 중인 서울 `대규모점포`를 시설 단위로 정리한 뒤 최근접 점포까지의 거리를 계산한다.

In [9]:
# ============================================================
# 서울 영업 중 대규모점포 추출
# ============================================================

mart_raw = pd.read_csv(
    MART_RAW_PATH,
    encoding="cp949",
)

seoul_mask = (
    mart_raw["도로명주소"].str.contains("서울특별시", na=False)
    | mart_raw["지번주소"].str.contains("서울특별시", na=False)
)

store_clean = (
    mart_raw.loc[
        seoul_mask
        & (mart_raw["영업상태명"] == "영업/정상")
        & (mart_raw["상세영업상태명"] == "정상영업")
        & (mart_raw["점포구분명"] == "대규모점포")
    ]
    .copy()
    .reset_index(drop=True)
)

print("원본:", mart_raw.shape)
print("서울 영업 중 대규모점포:", len(store_clean))

print("\n업태 분포:")
print(
    store_clean["업태구분명"]
    .value_counts(dropna=False)
)


원본: (1051, 26)
서울 영업 중 대규모점포: 407

업태 분포:
업태구분명
그 밖의 대규모점포    211
대형마트           48
쇼핑센터           38
복합쇼핑몰          31
시장             27
전문점            25
백화점            25
구분없음            2
Name: count, dtype: int64


In [10]:
# ============================================================
# 점포 좌표 변환 및 결측 좌표 보완
# EPSG:5174 → EPSG:4326
# ============================================================

store_clean["좌표정보(X)"] = pd.to_numeric(
    store_clean["좌표정보(X)"],
    errors="coerce",
)
store_clean["좌표정보(Y)"] = pd.to_numeric(
    store_clean["좌표정보(Y)"],
    errors="coerce",
)

store_transformer = Transformer.from_crs(
    "EPSG:5174",
    "EPSG:4326",
    always_xy=True,
)

source_coord_mask = (
    store_clean["좌표정보(X)"].notna()
    & store_clean["좌표정보(Y)"].notna()
)

lon, lat = store_transformer.transform(
    store_clean.loc[source_coord_mask, "좌표정보(X)"].to_numpy(),
    store_clean.loc[source_coord_mask, "좌표정보(Y)"].to_numpy(),
)

store_clean.loc[source_coord_mask, "경도"] = lon
store_clean.loc[source_coord_mask, "위도"] = lat

missing_idx = store_clean.index[
    ~get_valid_seoul_coord_mask(store_clean)
]

for idx in missing_idx:
    store_name = store_clean.loc[idx, "사업장명"]
    road_address = clean_address(store_clean.loc[idx, "도로명주소"])
    jibun_address = clean_address(store_clean.loc[idx, "지번주소"])

    lat, lon = None, None

    if road_address:
        lat, lon = geocode_address(
            road_address,
            KAKAO_REST_API_KEY,
        )

    if lat is None and jibun_address:
        lat, lon = geocode_address(
            jibun_address,
            KAKAO_REST_API_KEY,
        )

    if lat is None:
        lat, lon = geocode_keyword(
            f"서울 {store_name}",
            KAKAO_REST_API_KEY,
        )

    store_clean.loc[idx, "위도"] = lat
    store_clean.loc[idx, "경도"] = lon

print("좌표 보완 대상:", len(missing_idx))


좌표 보완 대상: 9


In [11]:
# ============================================================
# 대규모점포 시설 단위 정리
# ============================================================

address = (
    store_clean["도로명주소"]
    .fillna(store_clean["지번주소"])
)

store_clean["자치구"] = address.str.extract(
    r"서울특별시\s+(\S+구)"
)

before_rows = len(store_clean)

# 동일 사업장명·좌표 중복은 최신 레코드만 유지
store_clean = (
    store_clean
    .sort_values("데이터갱신시점", ascending=False)
    .drop_duplicates(
        subset=["사업장명", "위도", "경도"],
        keep="first",
    )
    .reset_index(drop=True)
)

# 동일 사업장·자치구에 정상 좌표 레코드가 있으면
# 좌표가 비정상인 중복 레코드는 제거
valid_name_district = (
    store_clean.loc[
        get_valid_seoul_coord_mask(store_clean),
        ["사업장명", "자치구"],
    ]
    .drop_duplicates()
    .assign(동일시설_정상좌표존재=True)
)

store_clean = store_clean.merge(
    valid_name_district,
    on=["사업장명", "자치구"],
    how="left",
)

duplicate_missing_mask = (
    store_clean["동일시설_정상좌표존재"].eq(True)
    & ~get_valid_seoul_coord_mask(store_clean)
)

store_clean = (
    store_clean.loc[~duplicate_missing_mask]
    .drop(columns="동일시설_정상좌표존재")
    .reset_index(drop=True)
)

# 수동 검증 결과 반영
wrong_tongil_mask = (
    store_clean["사업장명"].eq("통일상가")
    & store_clean["도로명주소"].str.contains(
        "장충단로13길 20",
        na=False,
    )
)

airport_market_mask = (
    store_clean["사업장명"].eq("(주)공항시장")
)

store_clean = (
    store_clean.loc[
        ~(wrong_tongil_mask | airport_market_mask)
    ]
    .reset_index(drop=True)
)

print("시설 정리 전:", before_rows)
print("시설 정리 후:", len(store_clean))


시설 정리 전: 407
시설 정리 후: 394


In [12]:
# ============================================================
# 대규모점포 최종 품질 검증
# ============================================================

valid_store_coord = get_valid_seoul_coord_mask(store_clean)

store_valid = (
    store_clean.loc[
        valid_store_coord,
        [
            "사업장명",
            "업태구분명",
            "위도",
            "경도",
        ],
    ]
    .reset_index(drop=True)
)

coord_duplicates = (
    store_valid
    .duplicated(
        subset=["사업장명", "위도", "경도"]
    )
    .sum()
)

print("최종 점포 수:", len(store_clean))
print("정상 서울 좌표:", len(store_valid))
print("비정상·결측 좌표:", (~valid_store_coord).sum())
print("동일 시설·좌표 중복:", coord_duplicates)


최종 점포 수: 394
정상 서울 좌표: 394
비정상·결측 좌표: 0
동일 시설·좌표 중복: 0


In [13]:
# ============================================================
# 최근접 대규모점포 피처 생성
# ============================================================

nearest_idx, nearest_distance = get_nearest_facility(
    df_living.loc[apt_coord_mask, "위도"].to_numpy(),
    df_living.loc[apt_coord_mask, "경도"].to_numpy(),
    store_valid["위도"].to_numpy(),
    store_valid["경도"].to_numpy(),
)

nearest_stores = (
    store_valid.iloc[nearest_idx]
    .reset_index(drop=True)
)

df_living.loc[
    apt_coord_mask,
    "최근접대규모점포",
] = nearest_stores["사업장명"].to_numpy()

df_living.loc[
    apt_coord_mask,
    "최근접대규모점포업태",
] = nearest_stores["업태구분명"].to_numpy()

df_living.loc[
    apt_coord_mask,
    "대규모점포거리_m",
] = nearest_distance

df_living.loc[
    df_living["대규모점포거리_m"] < 1,
    "대규모점포거리_m",
] = 0

df_living["대규모점포거리_km"] = (
    df_living["대규모점포거리_m"] / 1000
)


In [14]:
# ============================================================
# 대규모점포 피처 검증
# ============================================================

store_feature_cols = [
    "최근접대규모점포",
    "최근접대규모점포업태",
    "대규모점포거리_m",
    "대규모점포거리_km",
]

print("결측치:")
print(df_living[store_feature_cols].isna().sum())

print("\n거리 통계(km):")
print(df_living["대규모점포거리_km"].describe())


결측치:
최근접대규모점포      0
최근접대규모점포업태    0
대규모점포거리_m     0
대규모점포거리_km    0
dtype: int64

거리 통계(km):
count    234302.000000
mean          0.621204
std           0.400910
min           0.000000
25%           0.343741
50%           0.548411
75%           0.833438
max           3.492928
Name: 대규모점포거리_km, dtype: float64


# 3. 주요 일자리 접근성

`서울시 상권분석서비스(직장인구-행정동)`의 전 기간 평균 직장인구를 기준으로 **Top 10 행정동**을 주요 일자리 중심지로 정의한다.

각 행정동의 대표 좌표를 사용해 아파트에서 가장 가까운 주요 일자리 중심지까지의 거리를 계산한다.

In [15]:
# ============================================================
# 직장인구 및 행정동 영역 데이터 불러오기
# ============================================================

workplace_raw = pd.read_csv(
    WORKPLACE_PATH,
    encoding="cp949",
)

admin_area_raw = pd.read_csv(
    ADMIN_AREA_PATH,
    encoding="cp949",
)

workplace_raw["기준_년분기_코드"] = (
    workplace_raw["기준_년분기_코드"]
    .astype(str)
)
workplace_raw["행정동_코드"] = (
    workplace_raw["행정동_코드"]
    .astype(str)
)
admin_area_raw["행정동_코드"] = (
    admin_area_raw["행정동_코드"]
    .astype(str)
)

periods = sorted(
    workplace_raw["기준_년분기_코드"].unique()
)

workplace_codes = set(workplace_raw["행정동_코드"])
area_codes = set(admin_area_raw["행정동_코드"])

print("직장인구 데이터:", workplace_raw.shape)
print("기간:", periods[0], "~", periods[-1])
print("분기 수:", len(periods))
print("직장인구 행정동:", len(workplace_codes))
print("영역 행정동:", len(area_codes))
print("공통 행정동:", len(workplace_codes & area_codes))


직장인구 데이터: (8694, 24)
기간: 20211 ~ 20261
분기 수: 21
직장인구 행정동: 414
영역 행정동: 425
공통 행정동: 414


In [16]:
# ============================================================
# 주요 일자리 행정동 Top 10 선정
# ============================================================

job_top10 = (
    workplace_raw
    .groupby(
        ["행정동_코드", "행정동_코드_명"],
        as_index=False,
    )["총_직장_인구_수"]
    .mean()
    .rename(
        columns={
            "총_직장_인구_수": "평균_직장인구_수",
        }
    )
    .sort_values(
        "평균_직장인구_수",
        ascending=False,
    )
    .head(10)
    .reset_index(drop=True)
)

display(job_top10)


,행정동_코드,행정동_코드_명,평균_직장인구_수
0,11560540,여의동,211567.952381
1,11680580,삼성1동,178820.142857
2,11680640,역삼1동,172582.761905
3,11140550,명동,122300.761905
4,11110615,종로1?2?3?4가동,113868.809524
5,11545510,가산동,96786.000000
6,11140520,소공동,91348.619048
7,11680656,도곡2동,90693.523810
8,11680610,대치2동,86830.095238
9,11140540,회현동,77504.476190


In [17]:
# ============================================================
# 주요 일자리 중심좌표 결합 및 위경도 변환
# EPSG:5181 → EPSG:4326
# ============================================================

job_top10 = job_top10.merge(
    admin_area_raw[
        [
            "행정동_코드",
            "엑스좌표_값",
            "와이좌표_값",
        ]
    ],
    on="행정동_코드",
    how="left",
    validate="one_to_one",
)

if job_top10[
    ["엑스좌표_값", "와이좌표_값"]
].isna().any().any():
    raise ValueError("Top 10 일자리 행정동의 중심좌표가 누락되었습니다.")

job_transformer = Transformer.from_crs(
    "EPSG:5181",
    "EPSG:4326",
    always_xy=True,
)

job_top10["경도"], job_top10["위도"] = (
    job_transformer.transform(
        job_top10["엑스좌표_값"].to_numpy(),
        job_top10["와이좌표_값"].to_numpy(),
    )
)

display(
    job_top10[
        [
            "행정동_코드_명",
            "평균_직장인구_수",
            "위도",
            "경도",
        ]
    ]
)


,행정동_코드_명,평균_직장인구_수,위도,경도
0,여의동,211567.952381,37.528503,126.928466
1,삼성1동,178820.142857,37.515300,127.060798
2,역삼1동,172582.761905,37.500840,127.035499
3,명동,122300.761905,37.563951,126.984301
4,종로1?2?3?4가동,113868.809524,37.575187,126.989857
5,가산동,96786.000000,37.476660,126.883791
6,소공동,91348.619048,37.564256,126.974375
7,도곡2동,90693.523810,37.485492,127.049015
8,대치2동,86830.095238,37.500990,127.065661
9,회현동,77504.476190,37.556940,126.977105


In [18]:
# ============================================================
# 최근접 주요 일자리 거리 피처 생성
# ============================================================

nearest_idx, nearest_distance = get_nearest_facility(
    df_living.loc[apt_coord_mask, "위도"].to_numpy(),
    df_living.loc[apt_coord_mask, "경도"].to_numpy(),
    job_top10["위도"].to_numpy(),
    job_top10["경도"].to_numpy(),
)

nearest_jobs = (
    job_top10.iloc[nearest_idx]
    .reset_index(drop=True)
)

df_living.loc[
    apt_coord_mask,
    "최근접주요일자리",
] = nearest_jobs["행정동_코드_명"].to_numpy()

df_living.loc[
    apt_coord_mask,
    "주요일자리거리_m",
] = nearest_distance

df_living["주요일자리거리_km"] = (
    df_living["주요일자리거리_m"] / 1000
)


In [19]:
# ============================================================
# 주요 일자리 거리 피처 검증
# ============================================================

job_feature_cols = [
    "최근접주요일자리",
    "주요일자리거리_m",
    "주요일자리거리_km",
]

print("결측치:")
print(df_living[job_feature_cols].isna().sum())

print("\n거리 통계(km):")
print(df_living["주요일자리거리_km"].describe())

print("\n최근접 주요 일자리:")
print(
    df_living["최근접주요일자리"]
    .value_counts()
)


결측치:
최근접주요일자리      0
주요일자리거리_m     0
주요일자리거리_km    0
dtype: int64

거리 통계(km):
count    234302.000000
mean          5.317392
std           2.962110
min           0.016512
25%           3.006061
50%           4.790489
75%           7.234001
max          13.716420
Name: 주요일자리거리_km, dtype: float64

최근접 주요 일자리:
최근접주요일자리
종로1?2?3?4가동    63881
여의동            48019
삼성1동           30479
가산동            28068
대치2동           17740
소공동            15861
역삼1동           13887
명동              6549
회현동             5177
도곡2동            4641
Name: count, dtype: int64


# 4. 생활입지 피처 저장

학교·지하철 피처와 동일하게 `transaction_id`를 키로 하는 **feature-only parquet**으로 저장한다.

모델링 시 우선 사용할 핵심 수치 피처:
- `응급의료기관거리_km`
- `대규모점포거리_km`
- `주요일자리거리_km`

In [20]:
# ============================================================
# 생활입지 피처 데이터 구성
# ============================================================

living_feature_cols = [
    "transaction_id",

    # 응급의료기관
    "최근접응급의료기관",
    "최근접응급의료기관분류",
    "응급의료기관거리_m",
    "응급의료기관거리_km",

    # 대규모점포
    "최근접대규모점포",
    "최근접대규모점포업태",
    "대규모점포거리_m",
    "대규모점포거리_km",

    # 주요 일자리
    "최근접주요일자리",
    "주요일자리거리_m",
    "주요일자리거리_km",
]

df_fe_living = df_living[living_feature_cols].copy()

print("shape:", df_fe_living.shape)
print("\n결측치:")
print(df_fe_living.isna().sum())
print(
    "\ntransaction_id 중복:",
    df_fe_living["transaction_id"].duplicated().sum(),
)


shape: (234302, 12)

결측치:
transaction_id    0
최근접응급의료기관         0
최근접응급의료기관분류       0
응급의료기관거리_m        0
응급의료기관거리_km       0
최근접대규모점포          0
최근접대규모점포업태        0
대규모점포거리_m         0
대규모점포거리_km        0
최근접주요일자리          0
주요일자리거리_m         0
주요일자리거리_km        0
dtype: int64

transaction_id 중복: 0


In [21]:
# ============================================================
# parquet 저장
# ============================================================

df_fe_living.to_parquet(
    LIVING_FEATURE_PATH,
    index=False,
)

print("저장 완료:", LIVING_FEATURE_PATH)


저장 완료: c:\Users\uk\Apartment price\data\v2_5year_seoul\processed\df_fe_living.parquet
